## SINTETIZADOR DE CÓPULA GAUSSIANA

In [1]:
# Validar ambiente de execução do Python
import sys
print(sys.executable)

/home/alexandre/workspace/projects/incubator/synthetic-data-platform/.venv/bin/python


### 1. IMPORTAÇÕES E CONFIGURAÇÃO

In [2]:
# Importações e configuração

# Biblioteca padrão
import os
from pathlib import Path
from platform import python_version

# Manipulação e visualização de dados
import pandas as pd

# Geração e avaliação de dados sintéticos
from sdv.evaluation.single_table import (
    evaluate_quality,
    get_column_plot,
    run_diagnostic,
)
from sdv.io.local import CSVHandler
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

> **Nota sobre o kernel:** se uma importação do SDV/PyTorch for interrompida e surgir o erro `Only a single TORCH_LIBRARY ... triton`, reinicie o kernel antes de executar novamente. O PyTorch mantém registros nativos na memória que não podem ser corrigidos apenas repetindo o `import`.

In [3]:
# Exibir as versões das bibliotecas
%load_ext watermark
%watermark --iversions

pandas  : 2.3.3
platform: 1.0.8
sdv     : 1.37.0



In [4]:
# Exibir a versão do Python
print('A versão python utilizada neste jupyter notebook:', python_version())

A versão python utilizada neste jupyter notebook: 3.10.13


### 2. CARREGAR DADOS

O `CSVHandler` é o conector oficial do SDV para arquivos CSV. Ele lê uma ou mais tabelas e devolve um dicionário no formato `{nome_da_tabela: DataFrame}`. Por padrão, também preserva como texto os valores numéricos que começam com zero.

In [5]:
# Configurar o diretório do projeto
project_root = Path.cwd()

if project_root.name == "notebooks":
    os.chdir(project_root.parent)

print(f"Diretório do projeto: {Path.cwd().name}")

Diretório do projeto: synthetic-data-platform


In [6]:
# Definir os parâmetros de entrada e saída
input_directory = Path("data/processed/")
output_directory = Path("data/synthetic")

input_file = "customers.csv"
table_name = Path(input_file).stem
encoding = "utf-8"

In [7]:
# Carregar o conjunto de dados processado
input_path = input_directory / input_file

if not input_path.is_file():
    raise FileNotFoundError(f"Arquivo de entrada não encontrado: {input_path.resolve()}")

csv_connector = CSVHandler()
tables = csv_connector.read(
    folder_name=input_directory,
    file_names=[input_file],
    read_csv_parameters={"encoding": encoding},
)
dataset = tables[table_name]

### 3. INSPECIONAR DADOS

In [8]:
# Inspecionar a estrutura do conjunto de dados
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99244 entries, 0 to 99243
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   firstname            99244 non-null  object 
 1   lastname             99244 non-null  object 
 2   email                99244 non-null  object 
 3   address              99244 non-null  object 
 4   country              99244 non-null  object 
 5   last_country_logged  99244 non-null  object 
 6   creation_date        99244 non-null  object 
 7   last_activity_date   99244 non-null  object 
 8   age_group            99244 non-null  float64
 9   id                   99244 non-null  object 
dtypes: float64(1), object(9)
memory usage: 7.6+ MB


In [9]:
# Visualizar os primeiros registros
dataset.head()

,firstname,lastname,email,address,country,last_country_logged,creation_date,last_activity_date,age_group,id
0,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,02-17-2023 00:00:00,03-02-2023 00:43:50,4.0,280a5c38-422b-4f83-b472-5bac6450b11a
1,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,04-23-2022 00:00:00,03-02-2023 00:43:50,1.0,07de7b22-b1a1-4fa0-870e-67fbafc3b347
2,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,05-28-2021 00:00:00,03-02-2023 00:43:50,10.0,f97b472d-b859-450e-b053-87824e3ad5e0
3,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,08-26-2021 00:00:00,03-02-2023 00:43:50,3.0,c892ddd3-54d0-42cc-a784-98196241ff60
4,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,11-19-2022 00:00:00,03-02-2023 00:43:50,8.0,d81a15a5-2d1b-41d0-b0c3-0f5ff3322c49


### 4. CRIAR E CONFIGURAR OS METADADOS

In [10]:
# Criar metadados da tabela a partir do DataFrame
metadata = Metadata.detect_from_dataframe(
    data=dataset,
    table_name=table_name,
)

In [11]:
# Inspecionar os tipos detectados para cada coluna
metadata_dict = metadata.to_dict()
column_metadata = metadata_dict["tables"][table_name]["columns"]
pd.DataFrame.from_dict(column_metadata, orient="index")

,pii,sdtype,datetime_format
firstname,True,first_name,NaN
lastname,True,last_name,NaN
email,True,email,NaN
address,NaN,categorical,NaN
country,NaN,categorical,NaN
last_country_logged,NaN,categorical,NaN
creation_date,NaN,datetime,%m-%d-%Y %H:%M:%S
last_activity_date,NaN,datetime,%m-%d-%Y %H:%M:%S
age_group,NaN,numerical,NaN
id,NaN,id,NaN


#### 4.1. CORRIGIR OS TIPOS DETECTADOS

In [12]:
# Informar que as duas colunas armazenam códigos de países
metadata.update_column(column_name="country", sdtype="country_code")
metadata.update_column(
    column_name="last_country_logged",
    sdtype="country_code",
)

**Nota técnica**

As colunas `country` e `last_country_logged` foram ajustadas de `categorical` para `country_code`, pois armazenam códigos de países no padrão ISO Alpha-2. Esse ajuste permite que o SDV interprete corretamente o significado semântico desses campos. A coluna `address` foi mantida inalterada, uma vez que os dados utilizados neste projeto já são sintéticos.


#### 4.2. SALVAR OS METADADOS

In [13]:
# Salvar os metadados
metadata_file = output_directory / f"{table_name}_metadata.json"

output_directory.mkdir(parents=True, exist_ok=True)
metadata.save_to_json(metadata_file, mode="overwrite")

print(f"Metadados salvos em: {metadata_file}")

Metadados salvos em: data/synthetic/customers_metadata.json


### 5. CRIAR DADOS SINTÉTICOS

> **Atenção à privacidade:** neste exemplo, a chave primária original é preservada para manter compatibilidade com relacionamentos entre tabelas. Portanto, essa coluna não é sintética e não deve conter um identificador pessoal ou sensível em um cenário real.

In [14]:
# Criar e treinar o sintetizador
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(dataset)

In [15]:
# Gerar a mesma quantidade de registros do conjunto original
# Reiniciar o estado de amostragem torna a primeira amostra reproduzível.
synthesizer.reset_sampling()
synthetic_dataset = synthesizer.sample(num_rows=len(dataset))

In [16]:
# Preservar a chave primária para manter os relacionamentos entre tabelas
primary_key = "id"
synthetic_dataset[primary_key] = dataset[primary_key].to_numpy(copy=True)

# Manter a mesma ordem de colunas do conjunto original
synthetic_dataset = synthetic_dataset.loc[:, dataset.columns]

In [17]:
# Visualizar os primeiros dados sintéticos
synthetic_dataset.head()

,firstname,lastname,email,address,country,last_country_logged,creation_date,last_activity_date,age_group,id
0,Chelsea,Hill,robertsonjason@example.org,USCGC Long\nFPO AA 06479,KZ,BO,09-24-2022 20:12:54,03-01-2023 10:49:41,4.0,280a5c38-422b-4f83-b472-5bac6450b11a
1,Vincent,Foster,xguzman@example.net,"986 Duarte Forge Suite 988\nJeremychester, TN ...",IE,KP,10-26-2022 04:22:11,03-01-2023 12:07:36,6.0,07de7b22-b1a1-4fa0-870e-67fbafc3b347
2,Maria,Alexander,suttontheresa@example.net,"282 Amanda Road Apt. 209\nMatthewview, GU 81248",TL,CY,11-11-2022 05:45:08,03-01-2023 05:14:34,1.0,f97b472d-b859-450e-b053-87824e3ad5e0
3,Bradley,Johnson,monica38@example.org,"1730 Wilson Ramp\nSouth Samuelhaven, UT 51080",EC,AF,11-17-2022 11:16:23,03-02-2023 02:32:40,6.0,c892ddd3-54d0-42cc-a784-98196241ff60
4,Jennifer,Clayton,sburton@example.net,"46939 Macias Bridge\nPort Danaville, IN 25730",BD,RS,08-06-2022 21:47:41,03-02-2023 05:24:42,3.0,d81a15a5-2d1b-41d0-b0c3-0f5ff3322c49


In [18]:
# Comparar as dimensões dos conjuntos de dados
print(f"Quantidade de registros originais: {dataset.shape[0]}")
print(f"Quantidade de registros sintéticos: {synthetic_dataset.shape[0]}")
print(f"Quantidade de colunas sintéticas: {synthetic_dataset.shape[1]}")

Quantidade de registros originais: 99244
Quantidade de registros sintéticos: 99244
Quantidade de colunas sintéticas: 10


In [19]:
# Contar os valores ausentes em cada coluna
synthetic_dataset.isna().sum()

firstname              0
lastname               0
email                  0
address                0
country                0
last_country_logged    0
creation_date          0
last_activity_date     0
age_group              0
id                     0
dtype: int64

In [20]:
# Confirmar valores, ordem, nome e tipo da coluna usada nos relacionamentos.
pd.testing.assert_series_equal(
    synthetic_dataset[primary_key],
    dataset[primary_key],
    check_dtype=True,
    check_names=True
)

print(f"Validação concluída: a coluna '{primary_key}' foi preservada.")

Validação concluída: a coluna 'id' foi preservada.


### 6. AVALIAR OS DADOS SINTÉTICOS

#### 6.1. EXECUTAR O DIAGNÓSTICO

In [21]:
# Executar o diagnóstico dos dados sintéticos
diagnostic = run_diagnostic(
    real_data=dataset,
    synthetic_data=synthetic_dataset,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 10/10 [00:00<00:00, 144.65it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 96.18it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



#### 6.2. MEDIR A QUALIDADE


In [22]:
# Avaliar a semelhança entre os dados reais e sintéticos
quality_report = evaluate_quality(
    real_data=dataset,
    synthetic_data=synthetic_dataset,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 40.85it/s]|
Column Shapes Score: 93.42%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 64.62it/s]|
Column Pair Trends Score: 10.08%

Overall Score (Average): 51.75%



In [23]:
# Detalhar a qualidade da distribuição de cada coluna
quality_report.get_details("Column Shapes")


,Column,Metric,Score
0,address,TVComplement,0.967625
1,creation_date,KSComplement,0.938505
2,last_activity_date,KSComplement,0.965681
3,age_group,KSComplement,0.864868


#### 6.3. VISUALIZAR AS DISTRIBUIÇÕES

Os gráficos abaixo comparam os dados reais e sintéticos por duas perspectivas complementares.

##### Frequência de cada grupo de idade

In [24]:
# Comparar os grupos de idade

fig = get_column_plot(
    real_data=dataset,
    synthetic_data=synthetic_dataset,
    column_name="age_group",
    metadata=metadata,
    plot_type="bar"
)

fig.show(renderer="plotly_mimetype")

##### Distribuição dos grupos de idade

Como `age_group` representa faixas numéricas, uma cópia dos metadados será ajustada apenas para produzir o gráfico de distribuição. Os metadados usados pelo sintetizador permanecem inalterados.

In [25]:
# Tratar age_group como número apenas para o gráfico de distribuição
age_group_plot_metadata_dict = metadata.to_dict()
age_group_plot_metadata_dict["tables"][table_name]["columns"]["age_group"] = {
    "sdtype": "numerical"
}
age_group_plot_metadata = Metadata.load_from_dict(
    age_group_plot_metadata_dict
)

In [26]:
# Comparar a distribuição dos grupos de idade
fig = get_column_plot(
    real_data=dataset,
    synthetic_data=synthetic_dataset,
    column_name="age_group",
    metadata=age_group_plot_metadata,
    plot_type="distplot"
)

fig.show(renderer="plotly_mimetype")

### 7. SALVAR OS RESULTADOS

O sintetizador treinado será salvo para reutilização. Na gravação do CSV, o conector recebe novamente um dicionário: cada chave define o nome do arquivo e cada valor contém a tabela que será gravada.

In [27]:
# Definir os arquivos de saída
synthesizer_file = output_directory / "gaussian_copula_synthesizer.pkl"
output_file = output_directory / f"{table_name}_synthetic.csv"

In [28]:
# Salvar o sintetizador treinado
synthesizer.save(filepath=synthesizer_file)
print(f"Sintetizador salvo em: {synthesizer_file.resolve()}")

Sintetizador salvo em: /home/alexandre/workspace/projects/incubator/synthetic-data-platform/data/synthetic/gaussian_copula_synthesizer.pkl


In [29]:
# Salvar os dados sintéticos em CSV
csv_connector.write(
    synthetic_data={output_file.stem: synthetic_dataset},
    folder_name=output_directory,
    mode="w",
    to_csv_parameters={"encoding": encoding},
)

if not output_file.is_file():
    raise FileNotFoundError(f"O arquivo não foi salvo em: {output_file.resolve()}")

print(f"Dados sintéticos salvos e confirmados em: {output_file.resolve()}")


Dados sintéticos salvos e confirmados em: /home/alexandre/workspace/projects/incubator/synthetic-data-platform/data/synthetic/customers_synthetic.csv


### FIM
